# exp047 public PF/Beam gate-only audit

## Contents

1. Setup and configuration
2. Input feature check
3. Run gate-only audit
4. Metrics and artifacts

## 1. Setup and configuration

This notebook runs an audit-only train-side surrogate for clipped PF/Beam gate candidates. It does not create a submission file.

In [ ]:
import json
from pathlib import Path

import pandas as pd

from gate_only_audit import get_nested, load_local_config, resolve_feature_path, run_audit
from settings import EXPERIMENT_NAME, ExperimentPaths

paths = ExperimentPaths()
config = load_local_config()
feature_path = resolve_feature_path(paths, get_nested(config, 'data.feature_path'))

print('experiment:', EXPERIMENT_NAME)
print('route:', get_nested(config, 'experiment.route'))
print('status:', get_nested(config, 'experiment.status'))
print('feature_path:', feature_path)
print('split_systems:', get_nested(config, 'audit.split_systems'))
print('reference:', get_nested(config, 'audit.reference_control'))
print('fixed_gates:', [item['name'] for item in get_nested(config, 'audit.fixed_gates', [])])
print('gate_models:', [item['name'] for item in get_nested(config, 'audit.gate_models', [])])

## 2. Input feature check

In [ ]:
preview = pd.read_csv(feature_path, nrows=5)
print('columns:', len(preview.columns))
print(preview[['well_id', 'fold', 'cutoff_row', 'row_idx', 'eval_step', 'target_tvt', 'pf_pred', 'beam_pred']].head())

## 3. Run gate-only audit

In [ ]:
summary = run_audit(paths, config, feature_path)
print(json.dumps(summary, indent=2))

## 4. Metrics and artifacts

In [ ]:
artifact_files = sorted(paths.artifacts_dir.glob('public_pf_beam_gate_only_*'))
print('artifact_dir:', paths.artifacts_dir)
for path in artifact_files:
    print(path.name)

metrics = pd.read_csv(paths.artifacts_dir / 'public_pf_beam_gate_only_metrics.csv')
print(metrics.sort_values(['audit', 'rmse']).groupby('audit').head(8).to_string(index=False))

gate_stats = pd.read_csv(paths.artifacts_dir / 'public_pf_beam_gate_only_gate_stats.csv')
print(gate_stats.to_string(index=False))